# 工作坊 2：數據、預測與 R²
# Workshop 2: data, predictions and R²

先儲存自己的 Colab 副本。共有三節、五個程式格；依次按各格左邊的 ▶，等待表格或圖出現。每格可獨立執行。
Save your own Colab copy. There are three sections and five code cells. Run each cell in order and wait for its table or graph. Each cell runs independently.

以 `#` 開頭的行是說明用的註解。無需背誦程式或手算回歸。
Lines starting with `#` are explanatory comments. No memorising code or hand calculation is needed.

## 1. 數據探索 / Data exploration
外送店的模擬紀錄：`km` 是距離（公里），`fee` 是收費（港元）。相同位置是一對數據。每點是一筆紀錄。
Simulated delivery records: `km` is distance in kilometres; `fee` is the fee in HK dollars. Matching positions form a pair. Each dot is one record.

執行後，在工作紙描述關係，再找兩筆紀錄說明收費是否每次都隨距離增加。
Run the cell. Describe the relationship on your worksheet, then use two records to check whether the fee always rises with distance.

In [ ]:
# 載入畫圖工具 / Load the plotting tool
import matplotlib.pyplot as plt

# 每對數值是一筆外送紀錄 / Each pair is one delivery
km = [1, 2, 3, 4, 5, 6, 7, 8]
fee = [24, 29, 27, 38, 35, 44, 42, 51]

# 畫散點圖；每點是一筆紀錄 / Plot one dot per record
plt.figure(figsize=(8, 4))
plt.scatter(km, fee, color="#086c61", s=65)
# 標示座標軸及單位 / Label the axes and units
plt.xlabel("Delivery distance (km)")
plt.ylabel("Delivery fee (HK$)")
plt.title("Simulated delivery data")
plt.xlim(0, 9)
plt.ylim(0, 60)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 2. 線性回歸與模型表 / Linear regression and the model table
這是每天溫習時間與考試成績的模擬數據；每點是一位學生。執行本格，在工作紙抄下模型表的數值及每天 **4.5 小時**的預測成績。
These are simulated daily study times and exam scores; each dot represents a student. Run the cell and record the model-table values and prediction for **4.5 hours per day**.

- **n**：觀察數目。例子太少時，單靠這組數據不能建立很大信心。 / Number of observations. A small sample alone gives limited confidence.
- **m**：每多一小時，模型估計的分數改變多少。 / Change in estimated marks per extra hour.
- **c**：0 小時的估計分數。m、c 描述直線，不是準確度指標。 / Estimated marks at 0 hours. m and c describe the line; they do not measure accuracy.
差異指各個 y 值有高有低，例如學生的成績不同。 / Variation means that y-values differ, such as students having different scores.

- **R²（R square）**：直線解釋了這組成績差異的多少。較接近 1，這條線對建模數據的擬合較好；較接近 0，直線解釋的差異較少。 / How much variation in these scores is accounted for by the line. Nearer 1 means a better fit to the data used to build the model; nearer 0 means little variation is explained.

本組 R² ≈ 0.418 表示解釋了約 42% 的成績差異，並非 42% 的預測答對。高 R² 不保證新學生的成績能被準確預測；要用未參與建模的新數據，比較預測與實際結果。數據是模擬的，關聯不能證明因果。
Here, R² ≈ 0.418 means about 42% of the score variation is explained, not that 42% of predictions are correct. High R² does not guarantee reliable predictions for new students; use data not used to fit the model and compare predictions with actual results. These data are simulated and do not prove cause.

直線只畫在已觀察的 0–6 小時；座標軸保留 0–24 小時、0–100 分。
The line is drawn only over the observed 0–6 hours; axes retain 0–24 hours and 0–100 marks.

In [ ]:
# 載入工具 / Load the tools
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression

# 每對數值是一位模擬學生 / Each pair is a simulated student
hours = [0, 1, 2, 3, 4, 5, 6]
scores = [53, 46, 59, 52, 63, 54, 65]
X = [[h] for h in hours]
# fit 從數據找直線 / fit finds a line from the data
model = LinearRegression().fit(X, scores)
m = model.coef_[0]
c = model.intercept_
# R² 描述這條線對建模數據的擬合 / R² describes fit to these data
r_squared = model.score(X, scores)
# 用模型預測 4.5 小時 / Predict for 4.5 hours
study_hours = 4.5
prediction = model.predict([[study_hours]])[0]
# 顯示模型表；將結果寫在工作紙 / Display the table; record the results
summary = pd.DataFrame({
    "項目 / Metric": ["n", "m", "c", "R²", "4.5 h prediction"],
    "數值 / Value": [str(len(hours)), f"{m:.2f}", f"{c:.2f}",
                    f"{r_squared:.3f}", f"{prediction:.1f} marks"]
})
display(summary.set_index("項目 / Metric"))
# 在同一圖顯示數據、直線、預測 / Show data, line and prediction
plt.figure(figsize=(8, 4))
plt.scatter(hours, scores, color="#086c61", label="Data")
plt.plot([0, 6], model.predict([[0], [6]]),
         color="#b65320", label="Best-fit line")
plt.scatter(study_hours, prediction, color="#6b51a3",
            marker="D", s=85, label="Prediction")
plt.annotate(f"{study_hours} h: {prediction:.1f} marks",
             (study_hours, prediction), xytext=(8, 78),
             arrowprops=dict(arrowstyle="->", color="#6b51a3"))
# 完整時間、成績範圍 / Full time and score ranges
plt.xlim(0, 24)
plt.ylim(0, 100)
plt.xlabel("Daily study time (hours)")
plt.ylabel("Exam score (marks)")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 3. 三條線，哪條更能解釋數據？ / Which line explains its data better?
以下三組模擬數據包括較強的線性關聯、較弱的線性關聯、以及隨機數據；先不公布是哪一組。各組有 21 點，座標軸範圍相同。
These three simulated datasets include a stronger linear association, a weaker one and random data. Discover which is which. Each has 21 points and the same axis ranges.

每個例子是一個完整程式格。依次執行 A、B、C，查看散點、直線和模型表。在工作紙記下 R²，再按解釋各自數據差異的比例由高至低排列。
Each example is one complete code cell. Run A, B and C; inspect the dots, line and table. Record R², then rank how much variation each line explains in its own dataset.

固定隨機種子讓每次執行呈現相同例子。隨機數據也可能碰巧有小幅趨勢；R² 不一定剛好是 0。
Fixed random seeds keep the examples reproducible. Random data can show a small trend by chance; R² need not be exactly 0.

### 例子 A / Example A
執行整格，記下 R²。 / Run the whole cell and record R².

In [ ]:
# 例子 A：本格可獨立執行 / Example A: run this whole cell
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression

# 固定種子，保持例子相同 / Fix the seed to repeat the same example
rng = np.random.default_rng(2)
x = np.linspace(0, 10, 21)
y = 40 + 2*x + rng.normal(0, 14, len(x))
# 建立直線模型 / Fit the straight-line model
X = x.reshape(-1, 1)
model = LinearRegression().fit(X, y)
r_squared = model.score(X, y)
# 顯示模型表；比較時留意 R² / Show the model table; compare R²
summary = pd.DataFrame({
    "項目 / Metric": ["n", "m", "c", "R²"],
    "數值 / Value": [str(len(x)), f"{model.coef_[0]:.2f}",
                    f"{model.intercept_:.2f}", f"{r_squared:.3f}"]
})
display(summary.set_index("項目 / Metric"))
# 三個例子使用相同座標範圍 / Use the same axes for all examples
plt.figure(figsize=(8, 4))
plt.scatter(x, y, color="#086c61", label="Data")
plt.plot(x, model.predict(X), color="#b65320", label="Best-fit line")
plt.xlim(0, 10)
plt.ylim(0, 100)
plt.xlabel("x (input)")
plt.ylabel("y (output)")
plt.title("Example A")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 例子 B / Example B
執行整格，記下 R²。 / Run the whole cell and record R².

In [ ]:
# 例子 B：本格可獨立執行 / Example B: run this whole cell
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression

# 固定種子，保持例子相同 / Fix the seed to repeat the same example
rng = np.random.default_rng(12)
x = np.linspace(0, 10, 21)
y = 20 + 6*x + rng.normal(0, 3, len(x))
# 建立直線模型 / Fit the straight-line model
X = x.reshape(-1, 1)
model = LinearRegression().fit(X, y)
r_squared = model.score(X, y)
# 顯示模型表；比較時留意 R² / Show the model table; compare R²
summary = pd.DataFrame({
    "項目 / Metric": ["n", "m", "c", "R²"],
    "數值 / Value": [str(len(x)), f"{model.coef_[0]:.2f}",
                    f"{model.intercept_:.2f}", f"{r_squared:.3f}"]
})
display(summary.set_index("項目 / Metric"))
# 三個例子使用相同座標範圍 / Use the same axes for all examples
plt.figure(figsize=(8, 4))
plt.scatter(x, y, color="#086c61", label="Data")
plt.plot(x, model.predict(X), color="#b65320", label="Best-fit line")
plt.xlim(0, 10)
plt.ylim(0, 100)
plt.xlabel("x (input)")
plt.ylabel("y (output)")
plt.title("Example B")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 例子 C / Example C
執行整格，記下 R²。 / Run the whole cell and record R².

In [ ]:
# 例子 C：本格可獨立執行 / Example C: run this whole cell
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression

# 固定種子，保持例子相同 / Fix the seed to repeat the same example
rng = np.random.default_rng(27)
x = np.linspace(0, 10, 21)
y = rng.uniform(10, 90, len(x))
# 建立直線模型 / Fit the straight-line model
X = x.reshape(-1, 1)
model = LinearRegression().fit(X, y)
r_squared = model.score(X, y)
# 顯示模型表；比較時留意 R² / Show the model table; compare R²
summary = pd.DataFrame({
    "項目 / Metric": ["n", "m", "c", "R²"],
    "數值 / Value": [str(len(x)), f"{model.coef_[0]:.2f}",
                    f"{model.intercept_:.2f}", f"{r_squared:.3f}"]
})
display(summary.set_index("項目 / Metric"))
# 三個例子使用相同座標範圍 / Use the same axes for all examples
plt.figure(figsize=(8, 4))
plt.scatter(x, y, color="#086c61", label="Data")
plt.plot(x, model.predict(X), color="#b65320", label="Best-fit line")
plt.xlim(0, 10)
plt.ylim(0, 100)
plt.xlabel("x (input)")
plt.ylabel("y (output)")
plt.title("Example C")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

完成工作紙後分享：哪條線最能概括自己的數據？圖中的點支持你的排列嗎？
After completing the worksheet, discuss: which line best summarises its data? Do the dots support your ranking?

參考 / Reference: [scikit-learn R² documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html). 此處的 R² 均在建模數據上計算。 / All R² values here are computed on the data used to fit the model.